## Bounded Multi-dimensional Knapsack Problem
### Crop Production
How commercial farm models its seasonal planting decisions can be turned into a knapsack problem. The goal is to maximize total farm profit under strict constraints
1. <b>Objective Function</b>: Maximize the the total profit of producing crop $x_j$.
$$\text{maximize} \; Z = \sum_{j=1}^n p_j x_j - \sum_{j=1}^nF_jy_j$$ 
2. <b>Constraints</b>
* Knapsack Capacity: The farm has fixed, limited resources: arable lands, operating capital, and water allocation.
$$\sum_{j=1}^n a_{ij}x_j \leq R_i \quad \forall i$$
* Minimum and Maximum Bounds: The farmer cannot just plant the same crop across the total arable land since there is a bound to how much the contract buyer can only purchase.
$$ L_jy_j \geq x_j \geq Kjy_j \quad \forall j$$
* Variable Domains and Non-Negativity: Production quantities cannot be negative.
$$ x_j \geq 0 \quad \forall j$$
$$ y_j \in \{0, 1\} \quad \forall j$$
3. <b>Decision Variables</b>
* $x_j \ge 0 $: the exact amount of crop $j$ to plant.
* $y_j \in \{0,1\}$: 1 if crop $j$ is selected for production, and 0 if it is skipped.
4. <b>Parameters</b>
* $p_j$: the expected net profit generated per unit of crop $j$
* $a_{ij}$: the exact amount of resource $i$ required to produce one unit of crop $j$
* $R_i$: the total available limit of resource $i$
* $L_j$: the maximum quantity of crop $j$ that can be sold
* $F_j$: the production cost if crop $j$ is chosen
* $K_j$: the minimum viable production level required if crop $j$ is activated

#### Define the model name
Import `Model` from `docplex.mp.model` and create a `Model` object named <code>crop_production</code>.

In [1]:
import pandas as st

from docplex.mp.model import Model

mdl = Model(name="crop_production")

#### Read the CSV file containing the input data.
Read the two CSV files containing the crop production data and the resources data.

In [2]:
df = st.read_csv("crop_production_data.csv", encoding="utf-8-sig")

In [3]:
df

,crop_id,crop_name,profit_p_j,fixed_fee_F_j,min_bound_K_j,max_bound_L_j,req_land,req_capital,req_water
0,1,Rice,350,2000,15,150,1,200,5.0
1,2,Sweet corn,280,1500,10,120,1,140,2.5
2,3,Coconut,420,3500,8,70,1,250,1.5
3,4,Sugarcane,510,5000,20,100,1,450,4.5
4,5,Saba banana,480,4000,12,90,1,320,3.0


In [4]:
df_res = st.read_csv("resource_limits_data.csv", encoding="utf-8-sig")

In [5]:
df_res

,resource_id,resource_name,limit_R_i
0,1,Arable Land (acres),300
1,2,Operating Capital (_),60000
2,3,Water Allocation (acre-feet),800


#### Define the Parameters
Extract the following parameters: $p$, $F$, $K$, $L$, and $R$.

In [6]:
crops = df['crop_id'].tolist()
resources = df_res['resource_id'].tolist()

In [7]:
p = dict(zip(df["crop_id"], df["profit_p_j"]))                                                                       
F = dict(zip(df["crop_id"], df["fixed_fee_F_j"]))                                                                    
K = dict(zip(df["crop_id"], df["min_bound_K_j"]))                                                                    
L = dict(zip(df["crop_id"], df["max_bound_L_j"]))            
R = dict(zip(df_res["resource_id"], df_res["limit_R_i"])) 

Map the resource requirement for resource $i$ of crop $j$.

In [8]:
a = {}                                               
for _, row in df.iterrows():                                                                                         
    j = int(row["crop_id"])                                                                                          
    a[(1, j)] = row["req_land"]     # Resource 1: Arable Land                                                        
    a[(2, j)] = row["req_capital"]  # Resource 2: Operating Capital                                                  
    a[(3, j)] = row["req_water"]    # Resource 3: Water Allocation

#### Define the Decision Variables
Continuous Variable $x$

In [9]:
x = mdl.integer_var_dict(crops, lb=0, name="hectares_planted")

Binary variable $y$ for crop selection

In [10]:
y = mdl.binary_var_dict(crops, name="is_selected")

#### Define the Objective Function

In [11]:
total_profit = mdl.sum(p[j] * x[j] for j in crops)
total_fixed_cost = mdl.sum(F[j] * y[j] for j in crops)                                                  
mdl.maximize(total_profit - total_fixed_cost)

#### Define the Constraints
Knapsack Capacity Constraint

In [12]:
mdl.add_constraints(
    (mdl.sum(a[i, j] * x[j] for j in crops) <= R[i]) for i in resources
)

[docplex.mp.LinearConstraint[](hectares_planted_1+hectares_planted_2+hectares_planted_3+hectares_planted_4+hectares_planted_5,LE,300),
 docplex.mp.LinearConstraint[](200hectares_planted_1+140hectares_planted_2+250hectares_planted_3+450hectares_planted_4+320hectares_planted_5,LE,60000),
 docplex.mp.LinearConstraint[](5hectares_planted_1+2.500hectares_planted_2+1.500hectares_planted_3+4.500hectares_planted_4+3hectares_planted_5,LE,800)]

Minimum and Maximum Bound Constraints 

In [13]:
for j in crops:                                                                                                      
    # Minimum bound constraint: x_j >= K_j * y_j                                                                     
    mdl.add_constraint(x[j] >= K[j] * y[j], ctname=f"min_bound_crop_{j}")                                            
                                                                                                                        
    # Maximum bound constraint: x_j <= L_j * y_j                                                                     
    mdl.add_constraint(x[j] <= L[j] * y[j], ctname=f"max_bound_crop_{j}") 

#### Solve the Model

In [14]:
print("Zach is solving.....")
solution = mdl.solve(log_output=True)

Zach is solving.....
Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve modified 2 coefficients.
Reduced MIP has 13 rows, 10 columns, and 35 nonzeros.
Reduced MIP has 5 binaries, 5 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.02 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 13 rows, 10 columns, and 35 nonzeros.
Reduced MIP has 5 binaries, 5 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.02 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.00 sec. (0.02 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt    

#### Print Results
Aside from the model status, print Total Profit ($Z$) and print the following in a table:
* Selected crop ($y_j$)
* Hectares planted ($x_j$)
* Projected revenue($x_j * p_j$)
* Production cost ($F_j$)

In [17]:
if solution:                                                                                                         
    print("\n=== OPTIMAL SOLUTION FOUND ===")                                                                        
    print(f"Status: {mdl.get_solve_status()}")                                                                       
    print(f"Total Profit (Z): ${solution.objective_value:,.2f}")                                                     
                                             
    crop_names = dict(zip(df["crop_id"], df["crop_name"]))                                                           
                                                                                                                        
    selected_crops = []                                                                                              
    hectares = []                                                                                                    
    revenues = []                                                                                                    
    costs = []                                                                                                       
                                                                                                                        
    for j in crops:                                                                                                  
        # Check if crop j was selected                                                                               
        if solution.get_value(y[j]) > 0.5:                                                                           
            qty = solution.get_value(x[j])                                                                           
                                                                                                                        
            selected_crops.append(crop_names[j])                                                                     
            hectares.append(qty)                                                                                     
            revenues.append(qty * p[j])                                                                              
            costs.append(F[j])                                                                                       
                                                                                                                        
    # Construct the table                                                                                            
    results_df = st.DataFrame({                                                                                      
        'Selected Crop': selected_crops,                                                                             
        'Hectares Planted': hectares,                                                                                
        'Projected Revenue': revenues,                                                                               
        'Production Cost': costs                                                                                     
    })                                                                                                               
                                                                                                                        
else:                                                                                                                
    print("No feasible solution found.")


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Profit (Z): $92,870.00
